# sim_dual_arm — 本机 rclone 同步数据集与模型

本 Notebook 在**本机 Python 内核**运行，直接复用本机已配置好的 `rclone`（无需再次登录 Google Drive）。

## 使用流程

1. 在右上角 **Select Kernel** 选择**本机 Python 3 环境**。
2. 修改下方 **第 1 节** 的 `ROBOT` 与 `TASK`（例如 `marvin_sim` / `handover` 或 `marvin_sim` / `stack`）并运行。
3. 按需要单独运行后续功能格：
   - **第 2 节**：查看本地与云端数据集、模型及训练日志清单；
   - **第 3 节**：将本地 `var/<robot>/<task>/datasets/lerobot` 打包并上传到 Google Drive；
   - **第 4 节**：从 Google Drive 下载数据集回本地；
   - **第 5 节**：从 Google Drive 下载指定或最新训练模型与日志回本地；
   - **第 6 节**：将本地模型与日志上传备份至 Google Drive。


In [ ]:
# 1. 参数配置与环境初始化（修改 ROBOT / TASK 后运行）
import json
import shutil
import subprocess
from pathlib import Path

ROBOT = "marvin_sim"
TASK = "handover"
RCLONE_REMOTE = "gdrive:sim_dual_arm"
DATASET_ARCHIVE = "lerobot.tar"

def _find_repo_root() -> Path:
    for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (candidate / ".git").exists() and (candidate / "shared" / "scenarios").is_dir():
            return candidate
    raise FileNotFoundError(f"未从当前工作目录向上找到仓库根目录: {Path.cwd()}")

REPO_ROOT = _find_repo_root()
SCENARIO_DIR = REPO_ROOT / "shared" / "scenarios" / ROBOT / TASK
if not (SCENARIO_DIR / "binding.yaml").is_file():
    raise FileNotFoundError(f"方案不存在: {SCENARIO_DIR / 'binding.yaml'}")

if shutil.which("rclone") is None:
    raise RuntimeError("本机未找到 rclone，请先安装并配置好 gdrive remote。")

LOCAL_TASK_DIR = REPO_ROOT / "var" / ROBOT / TASK
REMOTE_TASK_DIR = f"{RCLONE_REMOTE}/{ROBOT}/{TASK}"

def run_rclone(args: list[str], *, capture: bool = False) -> subprocess.CompletedProcess:
    cmd = ["rclone", *args]
    if capture:
        return subprocess.run(cmd, check=True, capture_output=True, text=True)
    return subprocess.run(cmd, check=True)

def remote_files(remote_dir: str) -> set[str]:
    res = subprocess.run(["rclone", "lsf", remote_dir, "--files-only"], check=False, capture_output=True, text=True)
    if res.returncode != 0:
        return set()
    return {line.strip() for line in res.stdout.splitlines() if line.strip()}

def list_remote_dirs(remote_dir: str) -> list[str]:
    res = subprocess.run(["rclone", "lsf", remote_dir, "--dirs-only"], check=False, capture_output=True, text=True)
    if res.returncode != 0:
        return []
    return sorted(line.strip().rstrip("/") for line in res.stdout.splitlines() if line.strip())

def resolve_model_name(requested: str, candidates: list[str]) -> str:
    act_dirs = sorted((name for name in candidates if name.startswith("act_")), reverse=True)
    if requested == "latest":
        if not act_dirs:
            raise FileNotFoundError("未找到任何 act_* 模型目录。")
        return act_dirs[0]
    if requested not in candidates:
        raise FileNotFoundError(f"未找到指定模型目录 {requested!r}，可选: {act_dirs}")
    return requested

probe = subprocess.run(["rclone", "lsd", RCLONE_REMOTE], check=False, capture_output=True, text=True)
if probe.returncode != 0:
    raise RuntimeError(f"rclone 无法访问 {RCLONE_REMOTE}:\n{probe.stderr}")

print(f"仓库根目录: {REPO_ROOT}")
print(f"本地任务目录: {LOCAL_TASK_DIR}")
print(f"云端任务目录: {REMOTE_TASK_DIR}")
print("首次在新的 VS Code Colab 运行时注入凭证，可在本机终端复制 base64：")
print("  base64 -w0 ~/.config/rclone/rclone.conf")

In [ ]:
# 2. 查看本地与云端状态（只读）
local_ds = LOCAL_TASK_DIR / "datasets" / "lerobot"
local_manifest = local_ds / "manifest.json"
print("=== 本地数据集 ===")
if local_manifest.is_file():
    m = json.loads(local_manifest.read_text(encoding="utf-8"))
    splits = m.get("splits", {})
    tr, va = splits.get("training", {}), splits.get("validation", {})
    print(
        f"  {local_ds} | status={m.get('status')} | "
        f"train={tr.get('episode_count')}({tr.get('frame_count')}f) | "
        f"val={va.get('episode_count')}({va.get('frame_count')}f)"
    )
else:
    print(f"  (无) {local_ds}")

print("\n=== 云端数据集 ===")
res = subprocess.run(["rclone", "lsl", f"{REMOTE_TASK_DIR}/datasets"], check=False, capture_output=True, text=True)
print(res.stdout.strip() or "  (空或不存在)")

print("\n=== 本地模型 ===")
local_models_dir = LOCAL_TASK_DIR / "models"
local_models = sorted((p for p in local_models_dir.iterdir() if p.is_dir()), key=lambda p: p.name, reverse=True) if local_models_dir.is_dir() else []
if local_models:
    for p in local_models:
        sf = p / "model.safetensors"
        size_mb = sf.stat().st_size / (1024 * 1024) if sf.is_file() else 0.0
        print(f"  - {p.name} (model.safetensors={size_mb:.2f} MiB)")
else:
    print("  (无)")

print("\n=== 云端模型 ===")
remote_models = sorted((d for d in list_remote_dirs(f"{REMOTE_TASK_DIR}/models") if d.startswith("act_")), reverse=True)
if remote_models:
    for name in remote_models:
        print(f"  - {name}")
else:
    print("  (无)")

In [ ]:
# 3. 上传数据集（本地 var/<robot>/<task>/datasets/lerobot -> Google Drive）
local_datasets = LOCAL_TASK_DIR / "datasets"
local_lerobot = local_datasets / "lerobot"
if not (local_lerobot / "manifest.json").is_file():
    raise FileNotFoundError(f"未找到转换后的数据集: {local_lerobot / 'manifest.json'}")

tmp_tar = Path("/tmp") / f"{ROBOT}_{TASK}_{DATASET_ARCHIVE}"
remote_archive = f"{REMOTE_TASK_DIR}/datasets/{DATASET_ARCHIVE}"

try:
    print(f"正在打包数据集: {local_lerobot} -> {tmp_tar}")
    subprocess.run(["tar", "-C", str(local_datasets), "-cf", str(tmp_tar), "lerobot"], check=True)
    size_mb = tmp_tar.stat().st_size / (1024 * 1024)
    print(f"打包完成 ({size_mb:.2f} MiB)，开始上传至 {remote_archive}...")
    run_rclone(["mkdir", f"{REMOTE_TASK_DIR}/datasets"])
    run_rclone(["copyto", str(tmp_tar), remote_archive, "--progress"])
finally:
    if tmp_tar.exists():
        tmp_tar.unlink()

print("上传完成，云端文件信息:")
print(run_rclone(["lsl", remote_archive], capture=True).stdout.strip())

In [ ]:
# 4. 下载数据集（Google Drive -> 本地 var/<robot>/<task>/datasets/lerobot）
OVERWRITE_LOCAL = False  # 为防止误覆盖本地数据集，需覆盖时改为 True

local_datasets = LOCAL_TASK_DIR / "datasets"
target_dir = local_datasets / "lerobot"
if target_dir.exists() and not OVERWRITE_LOCAL:
    raise FileExistsError(f"本地数据集已存在: {target_dir}；若需覆盖请设 OVERWRITE_LOCAL = True")

local_datasets.mkdir(parents=True, exist_ok=True)
staging_dir = local_datasets / ".unpack_tmp"
if staging_dir.exists():
    shutil.rmtree(staging_dir)
staging_dir.mkdir(parents=True, exist_ok=True)
tmp_tar = Path("/tmp") / f"{ROBOT}_{TASK}_{DATASET_ARCHIVE}"
remote_datasets = f"{REMOTE_TASK_DIR}/datasets"

try:
    if DATASET_ARCHIVE in remote_files(remote_datasets):
        remote_archive = f"{remote_datasets}/{DATASET_ARCHIVE}"
        print(f"正在下载压缩包: {remote_archive} -> {tmp_tar}")
        run_rclone(["copyto", remote_archive, str(tmp_tar), "--progress"])
        subprocess.run(["tar", "-xf", str(tmp_tar), "-C", str(staging_dir)], check=True)
        if (staging_dir / "lerobot" / "manifest.json").is_file():
            extracted_root = staging_dir / "lerobot"
        elif (staging_dir / "manifest.json").is_file():
            extracted_root = staging_dir
        else:
            raise FileNotFoundError(f"{remote_archive} 解压后缺少 manifest.json")
    elif "manifest.json" in remote_files(f"{remote_datasets}/lerobot"):
        extracted_root = staging_dir / "lerobot"
        print(f"未找到 {DATASET_ARCHIVE}，直接同步远端目录: {remote_datasets}/lerobot")
        run_rclone(["copy", f"{remote_datasets}/lerobot", str(extracted_root), "--transfers", "8", "--progress"])
    else:
        raise FileNotFoundError(f"云端未找到数据集: {remote_datasets}")

    if target_dir.exists():
        shutil.rmtree(target_dir)
    shutil.move(str(extracted_root), str(target_dir))
finally:
    if tmp_tar.exists():
        tmp_tar.unlink()
    if staging_dir.exists():
        shutil.rmtree(staging_dir, ignore_errors=True)

m = json.loads((target_dir / "manifest.json").read_text(encoding="utf-8"))
print(f"数据集已就绪: {target_dir} | status={m.get('status')} | splits={list(m.get('splits', {}).keys())}")

In [ ]:
# 5. 下载模型与训练日志（Google Drive -> 本地 var/<robot>/<task>/models/）
MODEL_NAME = "latest"       # "latest" 自动选云端最新 act_*，也可填写具体目录名
SYNC_TRAINING_LOGS = True   # True 时同步拉取对应 logs/training/<model>/
OVERWRITE_LOCAL = False     # 若本地已存在同名模型目录，需覆盖时改为 True

remote_models = list_remote_dirs(f"{REMOTE_TASK_DIR}/models")
chosen = resolve_model_name(MODEL_NAME, remote_models)

local_model = LOCAL_TASK_DIR / "models" / chosen
if local_model.exists() and not OVERWRITE_LOCAL:
    raise FileExistsError(f"本地模型已存在: {local_model}；若需覆盖请设 OVERWRITE_LOCAL = True")

local_model.parent.mkdir(parents=True, exist_ok=True)
staging_model = local_model.parent / f".tmp_{chosen}"
if staging_model.exists():
    shutil.rmtree(staging_model)

try:
    remote_model = f"{REMOTE_TASK_DIR}/models/{chosen}"
    print(f"正在下载模型: {remote_model} -> {local_model}")
    run_rclone(["copy", remote_model, str(staging_model), "--transfers", "4", "--progress"])
    if not (staging_model / "model.safetensors").is_file():
        raise FileNotFoundError(f"下载结果缺少 model.safetensors: {staging_model}")
    if local_model.exists():
        shutil.rmtree(local_model)
    shutil.move(str(staging_model), str(local_model))
finally:
    if staging_model.exists():
        shutil.rmtree(staging_model, ignore_errors=True)

if SYNC_TRAINING_LOGS and chosen in list_remote_dirs(f"{REMOTE_TASK_DIR}/logs/training"):
    local_log = LOCAL_TASK_DIR / "logs" / "training" / chosen
    local_log.mkdir(parents=True, exist_ok=True)
    run_rclone(["copy", f"{REMOTE_TASK_DIR}/logs/training/{chosen}", str(local_log), "--transfers", "4", "--progress"])
    print(f"训练日志已同步: {local_log}")

print(f"模型下载完成: {local_model}")
for item in sorted(local_model.iterdir()):
    if item.is_file():
        print(f"  - {item.name}: {item.stat().st_size / (1024 * 1024):.2f} MiB")
args_file = local_model / "training_args.yaml"
if args_file.is_file():
    print("\n--- training_args.yaml ---")
    print(args_file.read_text(encoding="utf-8").strip())

In [ ]:
# 6. 上传本地模型与日志备份到 Google Drive
MODEL_NAME = "latest"      # "latest" 自动选本地最新 act_*，也可填写具体目录名
SYNC_TRAINING_LOGS = True  # True 时同步上传对应 logs/training/<model>/

local_models_dir = LOCAL_TASK_DIR / "models"
local_dirs = sorted(p.name for p in local_models_dir.iterdir() if p.is_dir()) if local_models_dir.is_dir() else []
chosen = resolve_model_name(MODEL_NAME, local_dirs)
local_model = local_models_dir / chosen
if not (local_model / "model.safetensors").is_file():
    raise FileNotFoundError(f"本地模型目录缺少 model.safetensors: {local_model}")

remote_model = f"{REMOTE_TASK_DIR}/models/{chosen}"
run_rclone(["mkdir", f"{REMOTE_TASK_DIR}/models"])
print(f"正在上传模型: {local_model} -> {remote_model}")
run_rclone(["copy", str(local_model), remote_model, "--transfers", "4", "--progress"])

local_log = LOCAL_TASK_DIR / "logs" / "training" / chosen
if SYNC_TRAINING_LOGS and local_log.is_dir():
    remote_log = f"{REMOTE_TASK_DIR}/logs/training/{chosen}"
    run_rclone(["mkdir", f"{REMOTE_TASK_DIR}/logs/training"])
    run_rclone(["copy", str(local_log), remote_log, "--transfers", "4", "--progress"])
    print(f"训练日志已上传: {remote_log}")

print(f"模型上传完成: {remote_model}")